# Local School — free GPU runtime on Kaggle (demo only)

Runs **Ollama** on Kaggle's free GPU and exposes it through a temporary Cloudflare tunnel,
behind a token so only your Local School app can use it.

1. *Settings → Accelerator:* **GPU T4 x2** (or P100). *Settings → Internet:* **On**.
2. **Run all.** The last cell prints a URL and token.
3. Local School → **Admin → Runtime**: type *Ollama*, paste the URL and token, **Test connection**, **Save**.

Limits: sessions end after ~12 hours (or when idle), the URL changes every session, and the
weekly GPU quota is ~30 hours. Hosting a server may conflict with Kaggle's terms, so use this
for short demos with **synthetic data only**. Use Modal or the school box for anything longer.

In [ ]:
CHAT_MODEL = "qwen3:8b"          # fits a 16 GB T4
EMBED_MODEL = "nomic-embed-text"


In [ ]:
%%bash
set -e
curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "installed: $(ollama --version) / $(cloudflared --version)"


In [ ]:
import os, subprocess, time
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "30m", "OLLAMA_NUM_PARALLEL": "4"}
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(1)
for m in (CHAT_MODEL, EMBED_MODEL):
    print("pulling", m); subprocess.run(["ollama", "pull", m], check=True, capture_output=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)


In [ ]:
# A tiny token-checking proxy in front of Ollama (the tunnel URL is public).
import secrets, threading, asyncio
from aiohttp import web, ClientSession, ClientTimeout

TOKEN = secrets.token_urlsafe(24)

async def handle(request):
    if request.headers.get("Authorization") != f"Bearer {TOKEN}":
        return web.json_response({"error": "unauthorized"}, status=401)
    async with ClientSession(timeout=ClientTimeout(total=None)) as s:
        async with s.request(request.method, f"http://127.0.0.1:11434{request.path_qs}",
                             data=await request.read(),
                             headers={"Content-Type": request.headers.get("Content-Type", "application/json")}) as r:
            resp = web.StreamResponse(status=r.status, headers={"Content-Type": r.headers.get("Content-Type", "application/json")})
            await resp.prepare(request)
            async for chunk in r.content.iter_any():
                await resp.write(chunk)
            await resp.write_eof()
            return resp

def run_proxy():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    app = web.Application(client_max_size=64 * 1024 * 1024)
    app.router.add_route("*", "/{tail:.*}", handle)
    runner = web.AppRunner(app); loop.run_until_complete(runner.setup())
    loop.run_until_complete(web.TCPSite(runner, "127.0.0.1", 8080).start()); loop.run_forever()

threading.Thread(target=run_proxy, daemon=True).start()
print("proxy listening on :8080")


In [ ]:
import re, subprocess, time
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
time.sleep(3)
print("\nPaste into Local School → Admin → Runtime")
print("  Type:  Ollama")
print("  URL:  ", url)
print("  Token:", TOKEN)


In [ ]:
# Keep the session alive while you demo (interrupt to stop).
import time
while True:
    time.sleep(60)
